# Experiment 01: Adversarial validation

Card: [exp01_adversarial.md](../../experiments/exp01_adversarial.md)

In [ ]:
import os
import sys

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.data import read_processed

SMOKE = os.environ.get("SMOKE") == "1"
WEATHER = ["air_temperature", "dew_temperature", "sea_level_pressure", "wind_speed", "wind_direction"]
FEATURES = ["meter", "primary_use", "square_feet", "year_built", *WEATHER, "hour", "dayofweek"]
COLUMNS = ["meter", "primary_use", "square_feet", "year_built", *WEATHER, "timestamp", "local_timestamp"]
if SMOKE:
    N_ROWS = 5_000  # rows sampled per set
    N_TREES = 10
    FIGURE = None
else:
    N_ROWS = 500_000
    N_TREES = 200
    FIGURE = "../../figures/exp01_adversarial.png"

## Data

In [ ]:
def load(name, extra=()):
    df = read_processed(name, columns=COLUMNS + list(extra))
    return df.assign(
        meter=df.meter.astype("category"),
        hour=df.local_timestamp.dt.hour,
        dayofweek=df.local_timestamp.dt.dayofweek,
    )


train = load("train_merged.parquet", ["anomaly"])
test = load("test_merged.parquet")
train_clean = train[~train.anomaly]  # what the models are trained on
print(f"train rows: {len(train):,} ({len(train_clean):,} without anomalies)  |  test rows: {len(test):,}")


def adversarial(a, b, features=FEATURES):
    """AUC of LightGBM telling rows of `a` from rows of `b`, and each feature's share of the gain."""
    rows = pd.concat([a.sample(N_ROWS, random_state=0), b.sample(N_ROWS, random_state=0)], ignore_index=True)
    y = np.r_[np.zeros(N_ROWS), np.ones(N_ROWS)]
    X_tr, X_te, y_tr, y_te = train_test_split(rows[features], y, test_size=0.2, random_state=0, stratify=y)
    model = lgb.LGBMClassifier(n_estimators=N_TREES, num_leaves=63, random_state=0, verbose=-1).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    gain = pd.Series(model.booster_.feature_importance("gain"), index=features)
    return auc, (gain / gain.sum()).sort_values()

## Training data (2016) vs test (2017–2018)

In [ ]:
auc_all, gain_all = adversarial(train_clean, test)
aucs = {
    "all features": auc_all,
    "without weather": adversarial(train_clean, test, [f for f in FEATURES if f not in WEATHER])[0],
    "weather only": adversarial(train_clean, test, WEATHER)[0],
    **{f"{w} alone": adversarial(train_clean, test, [w])[0] for w in WEATHER},
}
display(pd.Series(aucs, name="AUC").round(3).to_frame())

## Handout split: Jan–Sep vs Oct–Dec 2016

In [ ]:
train_months = train_clean[train_clean.timestamp < "2016-10-01"]
val_months = train[train.timestamp >= "2016-10-01"]
auc_split, gain_split = adversarial(train_months, val_months)
auc_split_no_weather = adversarial(train_months, val_months, [f for f in FEATURES if f not in WEATHER])[0]
print(f"AUC, all features: {auc_split:.3f}  |  without weather: {auc_split_no_weather:.3f}")

## Feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gain_all.plot.barh(ax=axes[0], title=f"Train vs test: feature share of gain (AUC {auc_all:.2f})")
gain_split.plot.barh(ax=axes[1], title=f"Jan–Sep vs Oct–Dec: feature share of gain (AUC {auc_split:.2f})")
plt.tight_layout()
if FIGURE:
    fig.savefig(FIGURE, dpi=150, bbox_inches="tight")